# 03 Fairness: Is the model fair?
Research question 4: Does the model wrongly flag some groups more often, or miss some groups more often? And what drives its predictions?

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from fairlearn.metrics import MetricFrame, selection_rate, false_positive_rate, false_negative_rate, count
import shap

df = pd.read_csv('../data/data.csv', sep=';')
df.columns = df.columns.str.strip()
df['dropout'] = (df['Target'] == 'Dropout').astype(int)

CATEGORICAL = ['Marital status', 'Application mode', 'Course', 'Previous qualification',
               'Nacionality', "Mother's qualification", "Father's qualification",
               "Mother's occupation", "Father's occupation"]
SEM1 = [c for c in df.columns if c.startswith('Curricular units 1st sem')]
SEM2 = [c for c in df.columns if c.startswith('Curricular units 2nd sem')]
enroll_features = [c for c in df.columns if c not in ['Target', 'dropout'] + SEM1 + SEM2]
sem1_features = enroll_features + SEM1
CUTOFF = 0.35

**Notes:** Same data and features as the model notebook. I'm auditing the final model from 02: logistic regression after semester 1, with a cutoff of 0.35. New tools: **Fairlearn** measures errors by group, and **SHAP** explains which features drive predictions.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df[sem1_features], df['dropout'], test_size=0.2, stratify=df['dropout'], random_state=42)

cats = [c for c in sem1_features if c in CATEGORICAL]
nums = [c for c in sem1_features if c not in CATEGORICAL]
model = Pipeline([
    ('prep', ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cats),
        ('num', StandardScaler(), nums)])),
    ('model', LogisticRegression(max_iter=2000, class_weight='balanced')),
])
model.fit(X_train, y_train)
test_prob = model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= CUTOFF).astype(int)
print('Flagged:', test_pred.sum(), 'of', len(test_pred))

**Notes:** Rebuilt the same model with the same split (`random_state=42`), so results match notebook 02: 382 of 885 test students flagged. `sparse_output=False` gives a regular table instead of a compressed one, which SHAP needs later.

In [ ]:
COLLEGE = [2, 3, 4, 5, 40, 43, 44]  # parent education codes for a completed college degree

groups = pd.DataFrame(index=X_test.index)
groups['Gender'] = X_test['Gender'].map({0: 'Female', 1: 'Male'})
groups['Age group'] = pd.cut(X_test['Age at enrollment'], bins=[0, 20, 24, 100],
                             labels=['17-20', '21-24', '25+'])
groups['Scholarship'] = X_test['Scholarship holder'].map({0: 'No', 1: 'Yes'})
groups['Displaced'] = X_test['Displaced'].map({0: 'No', 1: 'Yes'})
groups['International'] = X_test['International'].map({0: 'No', 1: 'Yes'})
first_gen = ~(X_test["Mother's qualification"].isin(COLLEGE) |
              X_test["Father's qualification"].isin(COLLEGE))
groups['First-generation'] = first_gen.map({False: 'No', True: 'Yes'})
groups['First-generation'].value_counts()

**Notes:** I compare the model's errors across 6 groups from the plan. **First-generation** means neither parent has a completed college degree (codes 2, 3, 4, 5, 40, 43, 44 in the UCI codebook). `isin` checks whether a code is in that list, `|` means "or," and `~` flips True/False. 85% of test students are first-generation, which is common in this Portuguese dataset. There is no race column, so race can't be checked.

In [ ]:
metrics = {'students': count, 'flagged_rate': selection_rate,
           'false_alarm_rate': false_positive_rate, 'miss_rate': false_negative_rate}

overall = MetricFrame(metrics=metrics, y_true=y_test, y_pred=test_pred,
                      sensitive_features=groups['Gender']).overall
rows = []
for col in groups.columns:
    mf = MetricFrame(metrics=metrics, y_true=y_test, y_pred=test_pred,
                     sensitive_features=groups[col])
    t = mf.by_group.reset_index()
    t.columns = ['group'] + list(t.columns[1:])
    t.insert(0, 'attribute', col)
    rows.append(t)
fair = pd.concat(rows, ignore_index=True)
fair['students'] = fair['students'].astype(int)
for c in ['flagged_rate', 'false_alarm_rate', 'miss_rate']:
    fair[c] = (fair[c] * 100).round(1)
print('Overall:', (overall[['flagged_rate', 'false_alarm_rate', 'miss_rate']] * 100).round(1).to_dict())
fair

**Notes:** Fairlearn's `MetricFrame` calculates each metric separately for every group.
- **Flagged rate:** % of the group the model flags
- **False alarm rate:** of students who did NOT drop out, % wrongly flagged
- **Miss rate:** of students who DID drop out, % the model missed

**Overall:** 43% flagged, 22% false alarm rate, 12% miss rate.

**Takeaways:**
- **Age:** students 25+ get false alarms at 38%, vs. 16.5% for ages 17-20 (2.3x). But the model misses young dropouts more (21% vs. 5%). It over-flags older students and under-catches younger ones.
- **Scholarship:** the biggest gap. The model misses 41% of scholarship holders who drop out, vs. 9% of non-holders. Because scholarship holders usually stay, the model overlooks the ones who struggle.
- **Gender:** men get more false alarms (31% vs. 18%), and women's dropouts are missed more (15% vs. 10%).
- **First-generation:** false alarm rates are equal (22% both), and the miss rate is actually lower for first-gen students (11% vs. 19%). No sign of unfairness here.
- **Displaced:** similar across groups.
- **International:** only 26 test students (8 dropouts), too few to judge.

In [ ]:
os.makedirs('../charts', exist_ok=True)
BLUE, ORANGE, GRAY = '#2a78d6', '#eb6834', '#8a8984'

fair['label'] = fair['attribute'] + ': ' + fair['group'].astype(str)
plot = fair.iloc[::-1]
fig, axes = plt.subplots(1, 2, figsize=(10, 6), sharey=True)
for ax, col, color, title in [(axes[0], 'false_alarm_rate', BLUE, 'False alarm rate (%)'),
                              (axes[1], 'miss_rate', ORANGE, 'Miss rate (%)')]:
    bars = ax.barh(plot['label'], plot[col], color=color, height=0.6)
    ax.bar_label(bars, fmt='%.0f', padding=3, fontsize=9)
    overall_val = overall[col] * 100
    ax.axvline(overall_val, color=GRAY, linestyle='--', linewidth=1,
               label=f'All students: {overall_val:.0f}%')
    ax.set_title(title, loc='left', fontweight='bold')
    ax.set_xlim(0, 100)
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(frameon=False, loc='lower right')
fig.suptitle('Who does the model get wrong?', x=0.02, ha='left', fontweight='bold', fontsize=14)
fig.tight_layout()
fig.savefig('../charts/7_fairness.png', dpi=150)
plt.show()

**Takeaway:** The two panels show both kinds of mistakes side by side. The dashed lines mark the overall rates (22% false alarms, 12% misses), so bars far from the line show unequal treatment. The biggest outliers are false alarms for students 25+ (38%) and misses for scholarship holders (41%).

In [ ]:
prep = model.named_steps['prep']
lr = model.named_steps['model']
X_train_t = prep.transform(X_train)
X_test_t = prep.transform(X_test)
names = prep.get_feature_names_out()

explainer = shap.LinearExplainer(lr, X_train_t)
sv = explainer(X_test_t)

shap_df = pd.DataFrame(sv.values, columns=names)
original = [n.replace('num__', '') if n.startswith('num__')
            else n.replace('cat__', '').rsplit('_', 1)[0] for n in names]
importance = shap_df.T.groupby(original).sum().T.abs().mean().sort_values(ascending=False)
top = importance.head(10)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(top.index[::-1], top.values[::-1], color=BLUE, height=0.6)
ax.bar_label(bars, fmt='%.2f', padding=3, fontsize=9)
ax.set_title('Top 10 drivers of dropout risk (SHAP)', loc='left', fontweight='bold')
ax.set_xlabel('Average impact on risk score (mean |SHAP value|)')
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig('../charts/8_shap_top_drivers.png', dpi=150)
plt.show()

**Takeaway:** The #1 driver by far is semester 1 courses passed: more courses passed means much lower risk. Next are tuition up to date (paid means lower risk), courses enrolled in semester 1 (taking more courses while passing the same number means higher risk), and course of study.

**Fairness note:** gender (#10) and age are used directly by the model. That helps explain the gaps in Cell 4: the model gives men and older students higher risk scores, which leads to more false alarms for them. Unemployment rate (#5) is the national economy the year a student enrolled. That's the same for everyone in a cohort, so it's a cohort effect, not something about the student.

## Fairness summary
Overall the model is accurate (catches 88% of dropouts), but its mistakes are not spread evenly.
- **Over-flagged:** students 25+ (38% false alarm rate vs. 16.5% for 17-20) and men (31% vs. 18%).
- **Under-caught:** scholarship holders (41% of their dropouts missed vs. 9%) and students 17-20 (21% vs. 5%).
- **Fair:** first-generation and displaced students had similar error rates.
- **Why:** SHAP shows the model mostly relies on semester 1 results and tuition, but it also uses gender and age directly.

**What I'd do:** an advisor makes the final call and never tells a student they are "high risk." Check these gaps every semester. After Oct 12, test removing gender and age as features, or using different cutoffs by group, and see whether the gaps shrink without losing too many caught dropouts.

**Limitations:** no race column. Small groups (26 international students) can't be judged reliably. Results come from one Portuguese university.